In [39]:
import uuid
from datetime import datetime, timedelta, date
from collections import defaultdict
from tabulate import tabulate
import numpy_financial as npf


# Define constants for date handling
DEFAULT_START_DATE = datetime.today()
DEFAULT_END_DATE = datetime(2099, 12, 31)

# Utility functions
def generate_id():
    return str(uuid.uuid4())


In [40]:
class SecuritiesTransaction:
    def __init__(self, type, instrument, quantity, price):
        self.transaction_id = generate_id()
        self.date = datetime.now()
        self.type = type
        self.instrument = instrument
        self.quantity = quantity
        self.price = price

In [41]:
class Portfolio:
    def __init__(self, name, client):
        self.portfolio_id = generate_id()
        self.name = name
        self.client = client
        self.cash_accounts = {'EUR': CashAccount('EUR')} # type: ignore
        self.securities_account = SecuritiesAccount() # type: ignore
        self.transactions = []

    def add_cash_account(self, currency):
        if currency not in self.cash_accounts:
            self.cash_accounts[currency] = CashAccount(currency) # type: ignore
            return self.cash_accounts[currency]
        else:
            raise ValueError("Cash account for this currency already exists.")

    def deposit(self, amount, currency='EUR'):
        if currency in self.cash_accounts:
            self.cash_accounts[currency].deposit(amount)
        else:
            raise ValueError("No cash account exists for this currency.")

    def withdraw(self, amount, currency='EUR'):
        if currency in self.cash_accounts:
            self.cash_accounts[currency].withdraw(amount)
        else:
            raise ValueError("No cash account exists for this currency.")

    def buy_security(self, instrument, quantity, price):
        if self.cash_accounts[instrument.issue_currency].balance >= quantity * price:
            self.securities_account.add_holding(instrument, quantity, price)
            self.cash_accounts[instrument.issue_currency].withdraw(quantity * price)
            self.transactions.append(SecuritiesTransaction('buy', instrument, quantity, price))
        else:
            raise ValueError("Insufficient funds to purchase securities.")

    def sell_security(self, instrument, quantity, price):
        if self.securities_account.holdings[instrument.instrument_id]['quantity'] >= quantity:
            self.securities_account.reduce_holding(instrument, quantity)
            self.cash_accounts[instrument.issue_currency].deposit(quantity * price)
            self.transactions.append(SecuritiesTransaction('sell', instrument, quantity, price))
        else:
            raise ValueError("Insufficient securities to sell.")

In [42]:
class Client:
    def __init__(self, name):
        self.client_id = generate_id()
        self.name = name
        self.portfolios = []

    def add_portfolio(self, name):
        portfolio = Portfolio(name, self)
        self.portfolios.append(portfolio)
        return portfolio


In [43]:
class CashAccount:
    def __init__(self, currency):
        self.currency = currency
        self.balance = 0.0
        self.transactions = []

    def deposit(self, amount):
        self.balance += amount
        self.transactions.append(('deposit', amount))

    def withdraw(self, amount):
        if self.balance >= amount:
            self.balance -= amount
            self.transactions.append(('withdraw', amount))
        else:
            raise ValueError("Insufficient funds.")

In [44]:
class SecuritiesAccount:
    def __init__(self):
        self.holdings = defaultdict(lambda: {'quantity': 0, 'average_price': 0.0})

    def add_holding(self, instrument, quantity, purchase_price):
        holding = self.holdings[instrument.instrument_id]
        total_quantity = holding['quantity'] + quantity
        total_value = (holding['quantity'] * holding['average_price']) + (quantity * purchase_price)
        holding['quantity'] = total_quantity
        holding['average_price'] = total_value / total_quantity

    def reduce_holding(self, instrument, quantity):
        holding = self.holdings[instrument.instrument_id]
        if quantity <= holding['quantity']:
            holding['quantity'] -= quantity
            if holding['quantity'] == 0:
                del self.holdings[instrument.instrument_id]
        else:
            raise ValueError("Not enough quantity to sell.")

In [45]:
# aanpassen:
# - melden als het id al bestaat
# - moet nu steeds een opject aanmaken voor elk instrument, wil gewoon naar instrument(123) of zo kunnen wijzen

class Instrument:
    def __init__(self, instrument_id, description, type, minimum_purchase_value, smallest_trading_unit, issue_currency):
        self.instrument_id = instrument_id
        self.description = description
        self.type = type
        self.minimum_purchase_value = minimum_purchase_value
        self.smallest_trading_unit = smallest_trading_unit
        self.issue_currency = issue_currency
        self.prices = []

    def is_bond(self):
        return self.type == "bond"

    def add_price(self, date, price):
        self.prices.append((date, price))
        self.prices.sort()

    def get_price(self, date):
        last_price = None
        for price_date, price in self.prices:
            if price_date <= date:
                last_price = price
            else:
                break
        return last_price if last_price is not None else "N/A"
           
    def get_details(self):
        return {
            "ID": self.instrument_id,
            "Description": self.description,
            "Type": self.type,
            "Minimum Purchase Value": self.minimum_purchase_value,
            "Smallest Trading Unit": self.smallest_trading_unit,
            "Issue Currency": self.issue_currency
        }

In [46]:
class Bond(Instrument):
    def __init__(self, instrument_id, description, minimum_purchase_value, smallest_trading_unit, issue_currency, start_date, maturity_date, interest_rate, interest_payment_frequency):
        super().__init__(instrument_id, description, "bond", minimum_purchase_value, smallest_trading_unit, issue_currency)
        self.start_date = start_date
        self.maturity_date = maturity_date
        self.interest_rate = interest_rate
        self.interest_payment_frequency = interest_payment_frequency

    def calculate_accrued_interest(self, as_of_date):
        if self.interest_payment_frequency == "annual":
            years_since_start = (as_of_date.year - self.start_date.year) - (1 if as_of_date < date(as_of_date.year, self.start_date.month, self.start_date.day) else 0)
            last_payment_date = date(self.start_date.year + years_since_start, self.start_date.month, self.start_date.day)
        elif self.interest_payment_frequency == "at maturity":
            last_payment_date = self.start_date
        else:
            raise ValueError("Unsupported payment frequency")

        if last_payment_date > self.maturity_date:
            last_payment_date = self.maturity_date

        days_accrued = (as_of_date - last_payment_date).days
        daily_interest_rate = (self.interest_rate / 100) / 365
        accrued_interest = self.minimum_purchase_value * daily_interest_rate * days_accrued

        return accrued_interest

    def calculate_ytm(self, current_price, years_to_maturity):
        face_value = self.minimum_purchase_value
        coupon = self.interest_rate / 100 * face_value

        # Corrected use of npf.rate without 'type' parameter
        ytm = npf.rate(nper=years_to_maturity, pmt=-coupon, pv=-current_price, fv=face_value)
        return ytm

    def calculate_duration(self, current_price, years_to_maturity):
        face_value = self.minimum_purchase_value
        coupon = self.interest_rate / 100 * face_value
        ytm = self.calculate_ytm(current_price, years_to_maturity)

        # Duration formula adjustment for correct calculation
        cash_flows = [(coupon if i < years_to_maturity else coupon + face_value) for i in range(1, years_to_maturity + 1)]
        discount_factors = [(1 + ytm) ** i for i in range(1, years_to_maturity + 1)]
        discounted_cash_flows = [cf / df for cf, df in zip(cash_flows, discount_factors)]
        duration = sum(dcf * i for i, dcf in enumerate(discounted_cash_flows, 1)) / sum(discounted_cash_flows)

        return duration

In [47]:
# Create a new client
client = Client("Alice Johnson")

# Add a portfolio for the client
portfolio = client.add_portfolio("Alice's Investment Portfolio")
portfolio.add_cash_account("USD")

# Deposit money into the EUR and USD accounts
portfolio.deposit(500000, "EUR")
portfolio.deposit(100000, "USD")

print(f"Client ID: {client.client_id}, Client Name: {client.name}")
for account in portfolio.cash_accounts:
    print(f"Account Currency: {portfolio.cash_accounts[account].currency}, Balance: {portfolio.cash_accounts[account].balance}")


Client ID: cf382d9a-c1d9-455b-a87e-5f65d1e09360, Client Name: Alice Johnson
Account Currency: EUR, Balance: 500000.0
Account Currency: USD, Balance: 100000.0


In [48]:
# Define a new instrument
US123 = Instrument('123', "Apple Inc. Stock", "share", 10, 1, "USD")
US123.add_price(datetime(2023, 4, 1), 5)
US123.add_price(datetime.now(), 10)  # Set the current price of Apple stock

# Create an instance of the instrument
US234 = Instrument('234',"Microsoft Corp. Stock", "share", 5, 1, "USD")
US234.add_price(datetime.now(), 320)  
US234.add_price(datetime(2023, 4, 1), 280)
US234.add_price(datetime(2023, 4, 15), 285)
US234.add_price(datetime(2022, 5, 1), 290)

# Define a new bond
bond = Bond("BOND001", "10-year Government Bond", 1000, 1, "EUR", date(2021, 4, 28), date(2030, 1, 1), 1.5, "annual")
bond.add_price(datetime.now(), 1.01) 
#print(f"Accrued Interest as of {date.today()}: {bond.calculate_accrued_interest(date.today()):.2f} {bond.issue_currency}")


# Client buys 10 shares of Apple
try:
    portfolio.buy_security(US123, 1000, 10)
except ValueError as e:
    print(f'Buying not succesful: {e}')

# Client buys bonds
try:
    portfolio.buy_security(bond, 10000, 1.01)
except ValueError as e:
    print(f'Buying not succesful: {e}')


print(f"Remaining holdings of Apple Stock: {portfolio.securities_account.holdings[US123.instrument_id]['quantity']}")
print(f"New balance on {US123.issue_currency} cash account: {portfolio.cash_accounts[US123.issue_currency].balance}")


Remaining holdings of Apple Stock: 1000
New balance on USD cash account: 90000.0


In [49]:
print(f"EUR Account Balance before transactions: {portfolio.cash_accounts['EUR'].balance}")
print(f"USD Account Balance before transactions: {portfolio.cash_accounts['USD'].balance}")


# Withdraw money from the EUR account
try:
    portfolio.withdraw(500, "EUR")
except ValueError as e:
    print(f'Withdrawal not succesful: {e}')

# Client sells 5 shares of Apple
try:
    portfolio.sell_security(US123, 5, US123.get_price(datetime.now()))
except ValueError as e:
    print(f'Selling not succesful: {e}')

print(f"Remaining holdings of Apple Stock: {portfolio.securities_account.holdings[US123.instrument_id]['quantity']}")
print(f"Remaining holdings of Microsoft: {portfolio.securities_account.holdings[US234.instrument_id]['quantity']}")
print(f"Remaining holdings of bond: {portfolio.securities_account.holdings[bond.instrument_id]['quantity']}")
print(f"Accrued Interest as of {date.today()}: {bond.calculate_accrued_interest(date.today()):.2f} {bond.issue_currency}")

#print(f"Account Balance after buying and selling securities: {portfolio.cash_accounts[US123.issue_currency].balance}")

print(f"EUR Account Balance after transactions: {portfolio.cash_accounts['EUR'].balance}")
print(f"USD Account Balance after transactions: {portfolio.cash_accounts['USD'].balance}")


EUR Account Balance before transactions: 489900.0
USD Account Balance before transactions: 90000.0
Remaining holdings of Apple Stock: 995
Remaining holdings of Microsoft: 0
Remaining holdings of bond: 10000
Accrued Interest as of 2024-05-05: 0.29 EUR
EUR Account Balance after transactions: 489400.0
USD Account Balance after transactions: 90050.0


In [50]:
# Attempt to withdraw more than the balance in the GBP account
try:
    portfolio.withdraw(50, "USD")
except ValueError as e:
    print(f'Withdrawal not succesful: {e}')

In [51]:


# Retrieve the price of the stock for a specific date
try:
    price_on_april_20 = US234.get_price(datetime(2023, 4, 20))
    print(f"Price of Microsoft stock on April 20, 2023: ${price_on_april_20}")
except ValueError as e:
    print(f'Pricing not succesful: {e}')

# Retrieve the price of the stock for a date before any prices were recorded
try:
    price_before_recorded = US234.get_price(datetime(2023, 3, 25))
except ValueError as e:
    print(f'Pricing not succesful: {e}')


Price of Microsoft stock on April 20, 2023: $285


In [52]:
bond_ids = {'BOND001'}

# Prepare data for the table
table_data = []
for instrument_id, data in portfolio.securities_account.holdings.items():
    quantity = data['quantity']
    value = data['quantity'] * data['average_price']
    # Format average price based on whether the instrument is a bond
    if instrument_id in bond_ids:
        avg_price = f"{data['average_price'] * 100:.2f}%"
    else:
        avg_price = f"{data['average_price']:.2f}"
    
    table_data.append([instrument_id, quantity, avg_price, value])

# Generate and print the table
headers = ['Instrument', 'Aantal/nominaal', 'Gem aankoopprijs', 'Waarde']
print(tabulate(table_data, headers=headers, tablefmt='simple'))


Instrument      Aantal/nominaal  Gem aankoopprijs      Waarde
------------  -----------------  ------------------  --------
123                         995  10.00                   9950
BOND001                   10000  101.00%                10100
234                           0  0.00                       0


In [53]:
from tabulate import tabulate

# Assuming holdings is a dictionary with instrument_id as keys and each value is an instance of `Instrument`
holdings = {
    '123': Instrument('123', 'Apple Stock', 'Stock', 100, 1, 'USD'),
    'BOND001': Instrument('BOND001', 'Government Bond', 'Bond', 1000, 1, 'EUR'),
    '234': Instrument('234', 'Tesla Stock', 'Stock', 500, 1, 'USD')
}

# Collect all instrument details
instrument_details = [instrument.get_details() for instrument in holdings.values()]

# Prepare table data
table_data = [[details['ID'], details['Description'], details['Type'], details['Minimum Purchase Value'], details['Smallest Trading Unit'], details['Issue Currency']] for details in instrument_details]

# Print the table
headers = ['ID', 'Description', 'Type', 'Min Purchase Value', 'Smallest Trading Unit', 'Issue Currency']
print(tabulate(table_data, headers=headers, tablefmt='pretty'))


+---------+-----------------+-------+--------------------+-----------------------+----------------+
|   ID    |   Description   | Type  | Min Purchase Value | Smallest Trading Unit | Issue Currency |
+---------+-----------------+-------+--------------------+-----------------------+----------------+
|   123   |   Apple Stock   | Stock |        100         |           1           |      USD       |
| BOND001 | Government Bond | Bond  |        1000        |           1           |      EUR       |
|   234   |   Tesla Stock   | Stock |        500         |           1           |      USD       |
+---------+-----------------+-------+--------------------+-----------------------+----------------+


In [54]:

# Bereid de gegevens voor de tabel voor
table_data = []
for instrument_id, data in holdings.items():
    instrument = data['instrument']
    quantity = data['quantity']
    value = quantity * data['average_price']
    # Formatteer gemiddelde prijs op basis van het type instrument
    if instrument.is_bond():
        avg_price = f"{data['average_price'] * 100:.2f}%"
    else:
        avg_price = f"{data['average_price']:.2f}"

    table_data.append([instrument_id, quantity, avg_price, value])

# Genereer en print de tabel
headers = ['Instrument', 'Aantal/nominaal', 'Gem aankoopprijs', 'Waarde']
print(tabulate(table_data, headers=headers, tablefmt='simple'))


TypeError: 'Instrument' object is not subscriptable

In [17]:

# Client buys 10 shares of Apple
try:
    portfolio.buy_security(US123, 10, 20)
except ValueError as e:
    print(f'Buying not succesful: {e}')



In [18]:
# Client buys bonds
try:
    portfolio.buy_security(bond, 10000, 1.02)
except ValueError as e:
    print(f'Buying not succesful: {e}')



In [23]:
# Dictionary to simulate which IDs are bonds for the sake of formatting
# In a real application, this should be determined by checking the type of each instrument
bond_ids = {'BOND001'}

# Prepare data for the table
table_data = []
for instrument_id, data in holdings.items():
    quantity = data['quantity']
    # Format average price based on whether the instrument is a bond
    if instrument_id in bond_ids:
        avg_price = f"{data['average_price'] * 100:.2f}%"
    else:
        avg_price = f"{data['average_price']:.2f}"
    
    table_data.append([instrument_id, quantity, avg_price])

# Generate and print the table
headers = ['Instrument', 'Aantal/nominaal', 'Gem aankoopprijs']
print(tabulate(table_data, headers=headers, tablefmt='pretty'))

TypeError: 'Instrument' object is not subscriptable

In [38]:
print(data['quantity'])

TypeError: 'Instrument' object is not subscriptable

In [20]:
bond.calculate_ytm(1.0, 2)

23.768405600525107

In [21]:
bond.calculate_duration(1.0,2)

1.7320453698685947

In [22]:
bond.calculate_accrued_interest(date(2024,1,1))

10.191780821917808